# Chunking the Zoo and Analysing Weight StatisticsPhase two. Every checkpoint gets flattened, normalised, companded, padded and chunked into`[1904, 144]` arrays, with a manifest holding everything needed to invert the chain.Then we check whether the paper's motivating claim, that weights are heavy-tailed, actuallyholds for small conv nets.

In [1]:
import os
import torch
import warnings
import chunking as C
from tqdm import tqdm
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from torch.utils.data import DataLoader, Dataset
from constants import SEED, device, TOKENS_PER_SEQ
warnings.filterwarnings("ignore")

## Extract the checkpointsChunk size 144 is chosen so every kept conv filter divides evenly. `C_in` is always amultiple of 16, so `C_in * 9` is always a multiple of 144 and no conv layer needs padding.All three gates run inside the loop. A corrupted manifest reaching the VAE costs days,catching it here costs nothing.

In [23]:

rows = []
models_path = Path('res_models')

EXCLUDE_NAMES = {'backbone_a.pt'}

ckpt_paths = sorted(
    p for p in models_path.rglob('*.pt')
    if 'resume' not in p.name and p.name not in EXCLUDE_NAMES
)

for path in tqdm(ckpt_paths, desc='extracting'):
    ck = torch.load(path, map_location='cpu')

    '''strict load so a checkpoint that does not match the architecture fails here'''
    model = resnet20(num_classes=100)
    model.load_state_dict(ck['state_dict'], strict=True)
    sd = model.state_dict()

    chunks, mask, seq_index, meta = C.extract_model(
        sd, ck['model_id'], checkpoint_path=str(path), split_id=ck['split_id'],
        seed=ck['seed'], epoch=ck['epoch'], init_group=ck['init_group']
    )

    C.verify_alignment(meta)
    C.verify_coverage(meta, sd)
    C.verify_roundtrip(chunks, meta, sd, verbose=False)

    C.save(f'./zoo_chunks/{ck["model_id"]}', chunks, mask, seq_index, meta)
    rows.append(ck['model_id'])

assert len(rows) == len(set(rows)), 'duplicate model_id, checkpoints would overwrite each other'
print(f'saved {len(rows)} expert models to ./zoo_chunks')

extracting: 100%|██████████| 50/50 [01:48<00:00,  2.17s/it]

saved 50 expert models to ./zoo_chunks


### Backbone, separatelyKept out of the zoo. It is a reference model: the target for convs-only merges, the upperbound to compare against, and a clean held-out reconstruction test. Cleaner if the VAE neversees it.

In [ ]:
ck = torch.load('res_models/backbone_a.pt', map_location='cpu')
model = resnet20(num_classes=100)
model.load_state_dict(ck['state_dict'], strict=True)
sd = model.state_dict()

chunks, mask, seq_index, meta = C.extract_model(
    sd, 'backbone_a', checkpoint_path='res_models/backbone_a.pt',
    split_id='full', seed=ck['seed'], epoch=ck['epoch'], init_group='base_a'
)
C.verify_alignment(meta); C.verify_coverage(meta, sd)
C.verify_roundtrip(chunks, meta, sd, verbose=False)
C.save('./zoo_reference/backbone_a', chunks, mask, seq_index, meta)
print('backbone saved to ./zoo_reference/backbone_a')

backbone saved to ./zoo_reference/backbone_a


## The datasetOne item is one sequence of 16 chunks, not one model. `self.index` holds `(model_idx,row_start)` pairs so the arrays stay per-model while the DataLoader sees a flat integerindex.`row_layer` maps each row back to its layer, which is where the depth and stage conditioningcomes from.

In [2]:
class ResZoo(Dataset):
    def __init__(self, root_dir='./zoo_chunks', model_ids=None):
        self.root_dir = root_dir
        '''filter to an explicit split, never load the whole directory blindly'''
        if model_ids is None:
            self.splits = sorted(os.listdir(root_dir))
        else:
            self.splits = sorted(model_ids)

        self.ids, self.chunks_list, self.mask_list, self.seq_index_list, self.meta_list = [], [], [], [], []
        '''flat index of (model_idx, row_start) so one item is one sequence'''
        self.index = []

        for split in self.splits:
            path = os.path.join(root_dir, split)
            chunks, mask, seq_index, meta = self.load_split_chunk(path)
            m = len(self.ids)
            self.ids.append(split)
            self.chunks_list.append(chunks)
            self.mask_list.append(mask)
            self.seq_index_list.append(seq_index)
            self.meta_list.append(meta)

            '''one entry per sequence, tokens_per_seq rows each'''
            T = meta.tokens_per_seq
            for start in range(0, chunks.shape[0], T):
                self.index.append((m, start))

        self.tokens_per_seq = self.meta_list[0].tokens_per_seq
        self.chunk_size = self.meta_list[0].chunk_size

        '''map each row back to its layer, needed for conditioning embeddings'''
        self.row_layer = []
        for meta in self.meta_list:
            lut = np.zeros(meta.n_chunks_total, dtype=np.int64)
            for li, lm in enumerate(meta.layers):
                lut[lm.chunk_start:lm.chunk_end] = li
            self.row_layer.append(lut)

    def load_split_chunk(self, path):
        chunks, mask, seq_index, meta = C.load(path)
        return chunks, mask, seq_index, meta

    def __len__(self):
        '''one item is one sequence, not one model'''
        return len(self.index)

    def __getitem__(self, idx):
        m, start = self.index[idx]
        end = start + self.tokens_per_seq
        lm_ids = self.row_layer[m][start:end]
        meta = self.meta_list[m]
        '''depth and stage of the layer this sequence came from'''
        depth = meta.layers[int(lm_ids[0])].depth_index
        stage = meta.layers[int(lm_ids[0])].stage
        return {
            'chunks': torch.from_numpy(self.chunks_list[m][start:end]).float(),
            'mask': torch.from_numpy(self.mask_list[m][start:end]).bool(),
            'depth': torch.tensor(depth, dtype=torch.long),
            'stage': torch.tensor(stage, dtype=torch.long),
            'model_idx': torch.tensor(m, dtype=torch.long),
            'row_start': torch.tensor(start, dtype=torch.long),
        }

## Split the zooExperts 0 to 2 train, expert 3 validates, expert 4 is held out entirely. The five finalcheckpoints are merge subjects and stay out of VAE training, otherwise the encoder hasmemorised the exact weights we later merge.Whole experts get held out rather than scattered checkpoints. Consecutive epochs within atrajectory are near-duplicates, so splitting inside one would not really hold anything out.The assertions at the end are what catch that contamination.

In [6]:
train_model = list(range(3))
train_epoch = list(range(4, 37, 4))
validation_model = [3,]
merget_set_epoch = [40]
test_model = [4,]

def str_to_model_attr_ids(name):
    parts = name.split('_')
    if len(parts) != 3:
        return None
    model_id, seed, epoch = parts
    model_id = model_id.replace('split', '')
    seed = seed.replace('seed', '')
    epoch = epoch.replace('ep', '')
    return int(model_id), int(seed), int(epoch)

zoo_dir = './zoo_chunks'
chunk_list = os.listdir(zoo_dir)

training_set = []
validation_set = []
test_set = []
merge_set = []

for chunk in chunk_list:
    parsed = str_to_model_attr_ids(chunk)
    if parsed is None:
        '''skip anything not matching splitN_seedN_epNNN'''
        continue
    model_id, seed, epoch = parsed

    if model_id in train_model and epoch in train_epoch:
        training_set.append(chunk)
    elif model_id in validation_model and epoch in train_epoch:
        validation_set.append(chunk)
    elif model_id in test_model:
        test_set.append(chunk)


    if model_id in (train_model + validation_model + test_model) and epoch in merget_set_epoch:
        merge_set.append(chunk)

'''training and validation must never share a model, and merge subjects must never appear in training - these three assertions are what catch a silent contamination bug before it reaches the vae'''
assert set(training_set).isdisjoint(validation_set)
assert set(training_set).isdisjoint(merge_set)
assert set(validation_set).isdisjoint(merge_set)



In [5]:
train_dataset = ResZoo(root_dir=zoo_dir, model_ids=training_set)
validation_dataset = ResZoo(root_dir=zoo_dir, model_ids=validation_set)
test_dataset = ResZoo(root_dir=zoo_dir, model_ids=test_set)
merge_dataset = ResZoo(root_dir=zoo_dir, model_ids=merge_set)

print(f'train {len(training_set)}  val {len(validation_set)}  ', f'test {len(test_set)}  merge {len(merge_set)}')

train 27  val 9   test 10  merge 5


In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
validation_dataloader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=4)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
merge_dataloader = DataLoader(merge_dataset, batch_size=32, shuffle=False, num_workers=4)

## Moment analysisPer-layer mean, variance, skewness and excess kurtosis, computed on the training split only.Measured twice, once on the stored companded values and once decompanded back to rawz-scores. The first pass alone is misleading: `log1p` compresses tails hard enough to flipkurtosis negative, which looks like the weights are thin-tailed when they are not.Raw kurtosis runs 3 to 9 in the early layers and drops to near zero by `layer3`. Socompanding is doing real work up front and slightly over-compressing deeper down.

### The moment analysis

In [13]:
import numpy as np
from scipy import stats as sstats

layer_stats_companded = {}
layer_stats_raw = {}

for m, meta in enumerate(train_dataset.meta_list):
    for lm in meta.layers:
        vals = train_dataset.chunks_list[m][lm.chunk_start:lm.chunk_end].reshape(-1)
        vals = vals[:lm.L]        # drop any padding

        '''values as stored, i.e. after compand - what the previous pass measured'''
        layer_stats_companded.setdefault(lm.key, []).append(vals)

        '''invert compand to get back to raw z-scored space, isolates the transform's effect'''
        raw = C.decompand(torch.from_numpy(vals).float(), lm.companding_type).numpy()
        layer_stats_raw.setdefault(lm.key, []).append(raw)

print(f"{'key':<26} {'kurt(companded)':>16} {'kurt(raw z-score)':>18} {'skew(raw)':>10}")
print("-" * 74)
for key in layer_stats_companded:
    v_comp = np.concatenate(layer_stats_companded[key])
    v_raw  = np.concatenate(layer_stats_raw[key])
    print(f"{key:<26} {sstats.kurtosis(v_comp):>16.3f} "
         f"{sstats.kurtosis(v_raw):>18.3f} {sstats.skew(v_raw):>10.3f}")

key                         kurt(companded)  kurt(raw z-score)  skew(raw)
--------------------------------------------------------------------------
layer1.0.conv1.weight                 0.030              6.254     -0.173
layer1.0.conv2.weight                -0.205              4.222      0.015
layer1.1.conv1.weight                -0.211              4.168     -0.377
layer1.1.conv2.weight                -0.176              4.195      0.194
layer1.2.conv1.weight                -0.373              3.003      0.020
layer1.2.conv2.weight                -0.136              8.865     -0.993
layer2.0.conv1.weight                 0.271              4.879      0.107
layer2.0.conv2.weight                -0.265              4.067     -0.111
layer2.1.conv1.weight                -0.501              3.139      0.080
layer2.1.conv2.weight                -0.917              0.268      0.150
layer2.2.conv1.weight                -0.754              1.362      0.270
layer2.2.conv2.weight                

## Independent models, separatelySame chunking, different directory. They are merge subjects, never vae training data, sokeeping them out of `zoo_chunks` means the split logic above cannot accidentally pick themup.

In [ ]:
for name in ['indep_0', 'indep_1']:    path = f'res_models_independent/{name}.pt'    ck = torch.load(path, map_location='cpu')    model = resnet20(num_classes=100)    model.load_state_dict(ck['state_dict'], strict=True)    sd = model.state_dict()    chunks, mask, seq_index, meta = C.extract_model(        sd, ck['model_id'], checkpoint_path=path, split_id=ck['split_id'],        seed=ck['seed'], epoch=ck['epoch'], init_group=ck['init_group']    )    C.verify_alignment(meta)    C.verify_coverage(meta, sd)    C.verify_roundtrip(chunks, meta, sd, verbose=False)    C.save(f'./zoo_independent/{name}', chunks, mask, seq_index, meta)    print(f'{name} saved to ./zoo_independent/{name}')